# Talking to a Real VLM: Running an Open-Source Vision-Language Model

In lecture, we saw closed, cloud-hosted VLMs like GPT-4V, Gemini, and Claude. Those are excellent, but you never get to look "inside" them.

In this notebook, you'll load and run a **real, open-source Vision-Language Model** yourself — on a free GPU, right here in Google Colab. We'll use **SmolVLM2 (2.2B parameters)** from Hugging Face, a small but genuinely capable VLM, small enough to run comfortably on Colab's free GPU.

By the end of this notebook you will have:
- Turned on a free GPU in Colab
- Downloaded and run a 2.2-billion-parameter VLM
- Asked it real questions about real images
- Measured how fast (or slow) it actually is — which connects directly to the "Bringing AI to the Edge" material
- Tried to find a case where it gets things wrong

> **Note:** This notebook needs a GPU to run at a reasonable speed. It **will not work well on CPU** — follow Section 1 before running anything else.


## 1. Turn On the GPU

Google Colab gives you free access to a GPU (usually an NVIDIA T4), but it's off by default. Turn it on:

1. In the menu, go to **Runtime → Change runtime type**.
2. Under "Hardware accelerator," choose **T4 GPU** (or any available GPU).
3. Click **Save**.

Now run the cell below to confirm the GPU is actually available.


In [ ]:
!nvidia-smi

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected — go back to Runtime > Change runtime type and select a GPU.")


### Challenge 1

Look at the output of `!nvidia-smi`. Answer these for yourself (no need to write code):

1. How much GPU memory (in MB or GB) do you have available?
2. Compare that number to your own laptop or phone. Is it more or less than you expected?


## 2. Install the Libraries

We need a recent version of `transformers` (the library that loads and runs the model) and `accelerate` (which helps place the model efficiently on the GPU).


In [ ]:
!pip install -q -U transformers accelerate
!pip install num2words
print("Done installing.")


## 3. Download Some Sample Images

Just like in our Data Science notebook, we'll download images directly with code — no manual upload needed. We'll grab a couple of well-known sample images from the public COCO dataset, with a fallback in case one link is ever unreachable.


In [ ]:
import os
import urllib.request

IMAGE_URLS = {
    "dog.jpg": [
        "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg",
        "https://raw.githubusercontent.com/ultralytics/yolov5/master/data/images/zidane.jpg",
    ],
    "street_scene.jpg": [
        "https://raw.githubusercontent.com/ultralytics/yolov5/master/data/images/bus.jpg",
        "https://raw.githubusercontent.com/opencv/opencv/master/samples/data/fruits.jpg",
    ],
}

def download_with_fallback(urls, out_path):
    for url in urls:
        try:
            print(f"Trying: {url}")
            urllib.request.urlretrieve(url, out_path)
            print(f"  Saved to {out_path}")
            return True
        except Exception as e:
            print(f"  Failed ({e}), trying next mirror...")
    return False

for filename, urls in IMAGE_URLS.items():
    if not os.path.exists(filename):
        ok = download_with_fallback(urls, filename)
        if not ok:
            raise RuntimeError(f"Could not download {filename} from any source.")
    else:
        print(f"{filename} already downloaded.")

print("All sample images ready:", list(IMAGE_URLS.keys()))


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, filename in zip(axes, IMAGE_URLS.keys()):
    img = Image.open(filename)
    ax.imshow(img)
    ax.set_title(filename)
    ax.axis("off")
plt.show()


## 4. Load the Model

We're using **`HuggingFaceTB/SmolVLM2-2.2B-Instruct`** — a real, open-source VLM with **2.2 billion parameters**. That's tiny compared to something like GPT-4V (which is estimated to have hundreds of billions), but it's a genuine, working VLM that follows the exact same pattern from lecture:

**image → vision encoder → tokens → language model → answer**

The first time you run this cell, it will download the model's weights (a few gigabytes) — this can take a minute or two. After that, it's cached.


In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
).to("cuda")

print("Model loaded on:", next(model.parameters()).device)
print("Number of parameters:", sum(p.numel() for p in model.parameters()) / 1e9, "billion")


> If you see an error mentioning `bfloat16`, change `torch_dtype=torch.bfloat16` to `torch_dtype=torch.float16` in the cell above and re-run it.


## 5. Ask the VLM About an Image

Now for the exciting part. We build a **conversation** — the same structure you'd use to chat with any modern LLM — except one of the messages is an image instead of just text.


In [ ]:
conversation = [
    {
        "role": "user",
        "content": [
            {"type": "image", "path": "dog.jpg"},
            {"type": "text", "text": "What's in this image? Describe it in two sentences."},
        ],
    }
]

inputs = processor.apply_chat_template(
    conversation,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    processor_kwargs={"return_tensors": "pt"},
).to(model.device)

generated_ids = model.generate(**inputs, do_sample=False, max_new_tokens=100)
answer = processor.batch_decode(
    generated_ids[:, inputs["input_ids"].shape[1]:],
    skip_special_tokens=True,
)[0]

print("VLM's answer:")
print(answer)


### What just happened?

Compare this to the "How a VLM Sees an Image" diagram from lecture:

| Diagram step | What we just did in code |
|---|---|
| Image | `Image.open("dog.jpg")`, loaded via the conversation's `"path"` |
| Vision Encoder | Handled inside `processor` / `model` — turns the image into tokens |
| Combined Tokens | `apply_chat_template(...)` merges image tokens + your text tokens |
| Language Model | `model.generate(...)` |
| Answer | `processor.batch_decode(...)` |

You didn't have to build any of this by hand — but now you've seen every stage of the pipeline actually run.


## 6. Ask Several Questions

Let's wrap the code above into a reusable function, then ask a few different questions about the same image.


In [ ]:
def ask_vlm(image_path, question, max_new_tokens=100):
    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "image", "path": image_path},
                {"type": "text", "text": question},
            ],
        }
    ]
    inputs = processor.apply_chat_template(
        conversation,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        processor_kwargs={"return_tensors": "pt"},
    ).to(model.device)

    generated_ids = model.generate(**inputs, do_sample=False, max_new_tokens=max_new_tokens)
    answer = processor.batch_decode(
        generated_ids[:, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )[0]
    return answer.strip()


In [ ]:
questions = [
    "What breed does this dog look like?",
    "What color is the dog?",
    "Is this dog indoors or outdoors? How can you tell?",
]

for q in questions:
    print("Q:", q)
    print("A:", ask_vlm("dog.jpg", q))
    print("-" * 40)


### 🔨 Challenge 2

1. Write **three of your own questions** about `street_scene.jpg` and run them through `ask_vlm(...)`.
2. For at least one question, ask something that requires *counting* or *comparing* — VLMs are often weaker at precise counting than at general description. Did it get it right?


In [ ]:
# Your code here



## 7. Try Your Own Photo

This step uses Colab's file upload widget, so it only works inside Google Colab (not in a generic Jupyter install).


In [1]:
from google.colab import files

uploaded = files.upload()  # choose an image file from your computer
my_image_path = list(uploaded.keys())[0]
print("Uploaded:", my_image_path)

from PIL import Image
Image.open(my_image_path)


IndexError: list index out of range

In [ ]:
# Ask your own question about your own photo
print(ask_vlm(my_image_path, "What's in this image?"))


## 8. How Fast Is It, Really?

This connects directly to the "Bringing AI to the Edge" material: **bigger models are powerful, but they are expensive to run.** Let's actually measure that cost, in seconds and tokens per second, instead of just talking about it.


In [ ]:
import time

def timed_ask_vlm(image_path, question, max_new_tokens=100):
    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "image", "path": image_path},
                {"type": "text", "text": question},
            ],
        }
    ]
    inputs = processor.apply_chat_template(
        conversation,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        processor_kwargs={"return_tensors": "pt"},
    ).to(model.device)

    torch.cuda.synchronize()
    start = time.time()
    generated_ids = model.generate(**inputs, do_sample=False, max_new_tokens=max_new_tokens)
    torch.cuda.synchronize()
    elapsed = time.time() - start

    new_tokens = generated_ids.shape[1] - inputs["input_ids"].shape[1]
    answer = processor.batch_decode(
        generated_ids[:, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )[0]

    print(f"Answer: {answer.strip()}")
    print(f"Time: {elapsed:.2f} sec | New tokens: {new_tokens} | Tokens/sec: {new_tokens / elapsed:.1f}")
    return elapsed

_ = timed_ask_vlm("dog.jpg", "Describe this image in detail, including colors and positions.", max_new_tokens=150)


### 🔨 Challenge 3

1. Run `timed_ask_vlm` again with `max_new_tokens=30` instead of `150`. How much faster is it?
2. Look back at the "real-time perception" example from the Edge AI lecture (1 second per frame). Based on your tokens/sec number, could this exact model realistically run at several frames per second on a live camera feed? Why or why not?


In [ ]:
# Your code here



## 9. 🔨 Challenge 4: Try to Break It

Every model has limits. Try to find a case where SmolVLM2 gets something wrong or gives a confused answer. Some ideas:

- Ask it to count objects in a busy, cluttered image.
- Ask about small text inside an image.
- Ask a question that requires knowledge the model wouldn't have (e.g. "what year was this photo taken?").
- Try an ambiguous or trick question ("what's NOT in this image?").

Write your attempt below, and note whether it succeeded or failed.


In [ ]:
# Your code here



_Write a one-sentence note here about what you found._


## 10. Bonus: A Much Smaller Model

Hugging Face also released a **256-million-parameter** version of the same model family — about **9x smaller** than the one we've been using. Let's load it and compare speed and answer quality on the same question.

This is the "efficient architectures" idea from the Edge AI lecture, in action: sometimes you trade a bit of quality for a model that's small enough to run on much more limited hardware.


In [ ]:
SMALL_MODEL_ID = "HuggingFaceTB/SmolVLM2-256M-Video-Instruct"

small_processor = AutoProcessor.from_pretrained(SMALL_MODEL_ID)
small_model = AutoModelForImageTextToText.from_pretrained(
    SMALL_MODEL_ID,
    torch_dtype=torch.bfloat16,
).to("cuda")

print("Small model parameters:", sum(p.numel() for p in small_model.parameters()) / 1e6, "million")


In [ ]:
def ask_small_vlm(image_path, question, max_new_tokens=100):
    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "image", "path": image_path},
                {"type": "text", "text": question},
            ],
        }
    ]
    inputs = small_processor.apply_chat_template(
        conversation,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        processor_kwargs={"return_tensors": "pt"},
    ).to(small_model.device)

    torch.cuda.synchronize()
    start = time.time()
    generated_ids = small_model.generate(**inputs, do_sample=False, max_new_tokens=max_new_tokens)
    torch.cuda.synchronize()
    elapsed = time.time() - start

    answer = small_processor.batch_decode(
        generated_ids[:, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )[0]
    print(f"Small model answer: {answer.strip()}")
    print(f"Small model time: {elapsed:.2f} sec")

question = "What's in this image? Describe it in two sentences."
print("=== 2.2B model ===")
timed_ask_vlm("dog.jpg", question)
print()
print("=== 256M model ===")
ask_small_vlm("dog.jpg", question)


### 🔨 Challenge 5 (reflection)

1. Which model was faster? By roughly how much?
2. Did the smaller model's answer seem noticeably worse, or was it still pretty good?
3. If you were putting a VLM on a robot with very little compute (like the Jetson boards from lecture), which of these two models would you pick — and what would you be trading away?


In [ ]:
# Your notes here



## Summary

You just did, with your own hands, everything the lecture described:

| Concept from lecture | What you did |
|---|---|
| VLMs turn images into tokens | Ran a real vision encoder + language model pipeline |
| VLMs answer questions about images | Asked SmolVLM2 real questions and got real answers |
| Bigger models are powerful but expensive | Measured actual seconds and tokens/sec |
| Efficient architectures trade size for speed | Compared a 2.2B model against a 256M model |
| Every model has limits | Found a case where the model struggled |

### Next steps
- Try a different open-source VLM from Hugging Face (search "image-text-to-text" on huggingface.co/models) and see how it compares.
- Try quantizing this model (see the `bitsandbytes` library) to see the "quantization" technique from lecture in action, not just in theory.
- Bring this hands-on experience into your own capstone project — could a VLM like this one be part of your system?
